# Import

In [1]:
from pathlib import Path
import sys
import json

In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
sys.path.append(str(ROOT))

In [3]:
from src.chunks import read_pages
from src.bm25 import BM25Retriever,tokenize
from src.chroma import load_chroma
from src.retrieval import build_hybride_retriever
from src.evaluation import load_eval, comparer_poids

# Tests

In [4]:
chunks = read_pages(DATA / 'processed/chunks.jsonl')

In [5]:
K_RETRIEVER = 10        # par bm25 / chroma
K_EVAL = 5     # hit@5
weights_rrf = [.5, .5]


In [6]:
bm25_retriever = BM25Retriever.from_documents(chunks, tokenize, k=K_RETRIEVER)

chroma_retriever = load_chroma(collection_name='rapport_v0',
                               chroma_dir=DATA / 'index/chroma_v0').as_retriever(search_kwargs={'k': K_RETRIEVER})

hybride_retriever = build_hybride_retriever([bm25_retriever, chroma_retriever], weights_rrf)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
questions = load_eval(DATA / 'eval/questions.jsonl')

In [8]:
comparer_poids([0.5, 0.3, 0.2], bm25_retriever, chroma_retriever, questions, k=K_EVAL)

bm25           hit@5=50.0%  MRR=0.41  ratés=['q03', 'q04', 'q07', 'q08', 'q10', 'q11', 'q12', 'q17', 'q18', 'q20', 'q21', 'q23']
chroma         hit@5=83.3%  MRR=0.64  ratés=['q04', 'q11', 'q12', 'q21']
hyb 0.5/0.5    hit@5=79.2%  MRR=0.56  ratés=['q04', 'q07', 'q11', 'q12', 'q21']
hyb 0.3/0.7    hit@5=83.3%  MRR=0.65  ratés=['q04', 'q11', 'q12', 'q21']
hyb 0.2/0.8    hit@5=83.3%  MRR=0.65  ratés=['q04', 'q11', 'q12', 'q21']

Rates par tous : [('q04', 'tableau'), ('q11', 'reformulee'), ('q12', 'tableau'), ('q21', 'multilingue')]
